In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
EDA_DIR = os.getcwd()
EDA_DIR

'/Users/sachinrana/Development/quick-ml-classifier/notebooks'

In [3]:
PROJECT_DIR = os.chdir("..")
PROJECT_DIR = os.getcwd()
PROJECT_DIR

'/Users/sachinrana/Development/quick-ml-classifier'

In [4]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# pandas display options display all columns and rows
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
# -------------------------
# Split numerical and categorical columns
# -------------------------
num_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns

In [ ]:
cat_cols = df.select_dtypes(
    include=["object", "category", "bool"]
).columns

# Numerical Features

In [ ]:
# Summary statistics
df[num_cols].describe()

In [ ]:
# Correlation matrix
corr = df[num_cols].corr()

print(corr)

# Categorical Features

In [ ]:
for col in cat_cols:
    print(f"\n{col}")
    print("Unique values:", df[col].nunique())

In [ ]:
print(
    (df[cat_cols].isnull().mean() * 100)
    .sort_values(ascending=False)
)

In [ ]:
# cardinality
cardinality = (
    df[cat_cols]
    .nunique()
    .sort_values(ascending=False)
)

print(cardinality)

# Target Analysis

print(
    df[target]
    .value_counts(normalize=True)
)

# Numerical Feature Vs Target

In [ ]:
for col in num_cols:
    if col == target:
        continue

    print("\n", col)

    print(
        df.groupby(target)[col]
        .agg(["mean", "median", "std"])
    )

# Categorical Feature Vs Target

In [ ]:
for col in cat_cols:
    print("\n", col)

    print(
        pd.crosstab(
            df[col],
            df[target],
            normalize="index"
        )
    )

# Json parsing

In [7]:
payload = {
    "customer_id": "1234",
    "amount": 350.5,
    "device_type": "mobile",
    "merchant_category": "electronics",
    "status": "approved",
    "location": {
        "country": "US",
        "city": "Seattle"
    }
}

In [8]:
from enum import Enum
class DeviceType(Enum):
    MOBILE = "mobile"
    DESKTOP = "desktop"
    TABLET = "tablet"

class MerchantCategory(Enum):
    ELECTRONICS = "electronics"
    GROCERY = "grocery"
    TRAVEL = "travel"


class Status(Enum):
    APPROVED = "approved"
    DECLINED = "declined"
    PENDING = "pending"

In [18]:
MerchantCategory("electronics").value

'electronics'

In [9]:
from dataclasses import dataclass

@dataclass
class Location:
    country: str
    city: str

In [ ]:
@dataclass
class Transaction:
    customer_id: str
    amount: float
    device_type: DeviceType
    merchant_category: MerchantCategory
    status: Status
    location: Location

    @classmethod
    def from_dict(cls, d: dict) -> "Transaction":

        return cls(
            customer_id=d["customer_id"],
            amount=float(d["amount"]),
            device_type=DeviceType(d["device_type"]),
            merchant_category=MerchantCategory(
                d["merchant_category"]
            ),
            status=Status(d["status"]),
            location=Location(
                country=d["location"]["country"],
                city=d["location"]["city"]
            )
        )

In [19]:
from pydantic import BaseModel

In [20]:
class Transaction(BaseModel):
    customer_id: str
    amount: float
    device_type: DeviceType
    merchant_category: MerchantCategory
    status: Status
    location: Location

In [21]:
transaction = Transaction.model_validate(payload)

print(transaction)

customer_id='1234' amount=350.5 device_type=<DeviceType.MOBILE: 'mobile'> merchant_category=<MerchantCategory.ELECTRONICS: 'electronics'> status=<Status.APPROVED: 'approved'> location=Location(country='US', city='Seattle')


# Sklearn Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [ ]:
from lightgbm import LGBMClassifier

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        )
    )
])

# Hyperparm Search

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "model__num_leaves": [31, 63, 127],
    "model__max_depth": [4, 6, 8, -1],
    "model__learning_rate": [0.01, 0.05, 0.1]
}

search = RandomizedSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_iter=20,
    n_jobs=-1
)

search.fit(X, y)

# Save Pipeline

In [ ]:
import joblib

joblib.dump(
    pipeline,
    "fraud_pipeline.pkl"
)

# Load Pipeline

In [ ]:
pipeline = joblib.load(
    "fraud_pipeline.pkl"
)

pipeline.predict(new_data)

# Pipeline Steps

preprocessing_steps = self.pipeline[:-1]
X_transformed = preprocessing_steps.transform(X)

In [ ]:
# 2. Extract the raw fitted classifier object
raw_classifier = self.pipeline.named_steps['classifier']

# Shape Importances

In [ ]:
explainer = shap.TreeExplainer(raw_classifier)
shap_values = explainer.shap_values(X_transformed_matrix)

# LightGBM outputs a list of arrays for binary/multiclass settings.
# Index 1 corresponds to the positive class impact.
if isinstance(shap_values, list):
    shap_values = shap_values[1]

In [ ]:
# 4. Collapse local row-level SHAP arrays into Global Feature Importances
# We take the mean of the absolute values across all records
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

In [ ]:
# 5. Extract feature names and assemble summary report
feature_names = self.get_feature_names()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
})